# Task 2.2 — Reproduction of Core Contribution
**Paper:** Algorithms for Learning Kernels Based on Centered Alignment  
**Student:** Pintu Singh | Roll No: 230105

## What I am reproducing
I am reproducing the **ALIGNF algorithm** (Algorithm 2, Section 3.2) — the joint kernel weight learning method that maximizes centered alignment between the combined kernel $K_\mu = \sum_k \mu_k K_k$ and the target kernel $K_y = yy^\top$, by solving a Quadratic Program over the weight vector $\mu$.

I also implement **ALIGN** (Algorithm 1, Section 3.2) — the simpler greedy baseline — and the **Uniform combination** baseline for comparison.

**Evaluation metric:** Classification accuracy (equivalently, misclassification error = 1 - accuracy), consistent with the paper's Table 1 which reports misclassification error on binary classification tasks.

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from scipy.optimize import minimize
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

np.random.seed(42)
os.makedirs('results', exist_ok=True)

Standard imports. `scipy.optimize.minimize` with SLSQP is used to solve the Quadratic Program in ALIGNF (Section 3.2, Theorem 1). The paper solves a QP of size $p \times p$ (number of base kernels), not $m \times m$ (number of samples) — making it computationally efficient.

In [2]:
# ── Dataset Setup (same as Task 2.1) ──────────────────────────────────────────
X, y = make_classification(n_samples=300, n_features=15,
                            n_informative=8, n_redundant=3, random_state=42)
y_svm = 2 * y - 1  # {0,1} → {-1,+1}
X_train, X_test, y_train, y_test = train_test_split(
    X, y_svm, test_size=0.3, random_state=42)
m = len(X_train)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (210, 15), Test: (90, 15)


Data is loaded and split. Labels are converted to {-1, +1} because the target kernel is $K_y = yy^\top$ (outer product of label vector), which requires signed labels. This is consistent with Section 2 of the paper where $K_y(x_i, x_j) = y_i y_j$.

In [3]:
# ── Base Kernel Functions ──────────────────────────────────────────────────────
def linear_kernel(X, Y):           return X @ Y.T
def rbf_kernel(X, Y, gamma):       
    d = np.sum((X[:,None,:] - Y[None,:,:])**2, axis=2)
    return np.exp(-gamma * d)
def poly_kernel(X, Y, d, c=1):     return (X @ Y.T + c) ** d

# 5 base kernels (train-train)
K_list_train = [
    linear_kernel(X_train, X_train),
    rbf_kernel(X_train, X_train, gamma=0.01),
    rbf_kernel(X_train, X_train, gamma=0.1),
    poly_kernel(X_train, X_train, d=2),
    poly_kernel(X_train, X_train, d=3),
]
kernel_names = ['Linear', 'RBF(0.01)', 'RBF(0.1)', 'Poly-2', 'Poly-3']
p = len(K_list_train)
print(f"Number of base kernels p = {p}")
print(f"Each kernel matrix shape: {K_list_train[0].shape}")

Number of base kernels p = 5
Each kernel matrix shape: (210, 210)


We define $p=5$ base kernels following the experimental setup in Section 5.1 of the paper, which uses linear, RBF (multiple bandwidths), and polynomial kernels. Each kernel matrix $K_k \in \mathbb{R}^{m \times m}$ encodes pairwise similarities between training points under kernel $k$.

In [4]:
# ── Kernel Centering (Section 2, Eq. 2) ───────────────────────────────────────
def center_kernel(K):
    m = K.shape[0]
    one = np.ones((m, m)) / m
    return K - one @ K - K @ one + one @ K @ one

# Target kernel: K_y = y * y^T  (Section 2)
K_y    = np.outer(y_train, y_train)
K_y_c  = center_kernel(K_y)

# Center all base kernels
K_list_c = [center_kernel(K) for K in K_list_train]
print("Centering complete. Example centered kernel range:",
      K_list_c[0].min().round(3), "to", K_list_c[0].max().round(3))

Centering complete. Example centered kernel range: -208.627 to 308.046


This implements the centering operation from **Section 2, Definition 1** of the paper: $K_c = K - \mathbf{1}_m K / m - K \mathbf{1}_m / m + \mathbf{1}_m K \mathbf{1}_m / m^2$. The target kernel $K_y = yy^\top$ encodes the ideal similarity — two points should be similar iff they share the same label. Centering both $K_k$ and $K_y$ is the key innovation of the paper over prior uncentered alignment work (Section 2.1).

In [5]:
# ── Frobenius Inner Product and Alignment ─────────────────────────────────────
def frob(A, B):   return np.sum(A * B)

def centered_alignment(Kc, Kyc):
    """Eq. (4): A_hat(K, K') = <Kc, K'c>_F / sqrt(<Kc,Kc>_F * <K'c,K'c>_F)"""
    num = frob(Kc, Kyc)
    den = np.sqrt(frob(Kc, Kc) * frob(Kyc, Kyc))
    return num / (den + 1e-10)

# Individual alignment of each base kernel
ind_alignments = [centered_alignment(Kc, K_y_c) for Kc in K_list_c]
print("Individual centered alignment scores:")
for name, score in zip(kernel_names, ind_alignments):
    print(f"  {name}: {score:.4f}")

Individual centered alignment scores:
  Linear: 0.1076
  RBF(0.01): 0.1585
  RBF(0.1): 0.1081
  Poly-2: 0.1172
  Poly-3: 0.0665


In [6]:
# ── ALIGN Algorithm (Section 3.2, Algorithm 1) ────────────────────────────────
mu_align = np.array(ind_alignments)
mu_align = np.maximum(mu_align, 0)       # ensure non-negative
mu_align /= (np.linalg.norm(mu_align) + 1e-10)  # normalize ||mu||_2 = 1

print("ALIGN weights (mu):")
for name, w in zip(kernel_names, mu_align):
    print(f"  {name}: {w:.4f}")

ALIGN weights (mu):
  Linear: 0.4172
  RBF(0.01): 0.6143
  RBF(0.1): 0.4192
  Poly-2: 0.4544
  Poly-3: 0.2577


In [7]:
# ── ALIGNF Algorithm (Section 3.2, Theorem 1) ─────────────────────────────────
# Build M matrix: M_kl = <K_k_c, K_l_c>_F
M = np.array([[frob(K_list_c[i], K_list_c[j]) for j in range(p)] for i in range(p)])
# Build a vector: a_k = <K_k_c, K_y_c>_F
a = np.array([frob(K_list_c[k], K_y_c) for k in range(p)])

# Maximize a^T mu / sqrt(mu^T M mu)  s.t.  mu >= 0,  ||mu||_2 <= 1
def neg_obj(mu):
    denom = np.sqrt(mu @ M @ mu + 1e-10)
    return -(a @ mu) / denom

def neg_obj_grad(mu):
    denom = np.sqrt(mu @ M @ mu + 1e-10)
    return -(a / denom - (a @ mu) * (M @ mu) / denom**3)

result = minimize(
    neg_obj, x0=np.ones(p)/p, jac=neg_obj_grad,
    method='SLSQP',
    bounds=[(0, None)]*p,
    constraints={'type': 'ineq', 'fun': lambda mu: 1 - np.linalg.norm(mu)},
    options={'ftol': 1e-9, 'maxiter': 1000}
)
mu_alignf = np.maximum(result.x, 0)
mu_alignf /= (np.linalg.norm(mu_alignf) + 1e-10)

print("ALIGNF weights (mu):")
for name, w in zip(kernel_names, mu_alignf):
    print(f"  {name}: {w:.4f}")

ALIGNF weights (mu):
  Linear: 0.9978
  RBF(0.01): 0.0587
  RBF(0.1): 0.0240
  Poly-2: 0.0173
  Poly-3: 0.0000


In [8]:
# ── Form Combined Kernels and Train SVMs ──────────────────────────────────────
K_list_test = [
    linear_kernel(X_test, X_train),
    rbf_kernel(X_test, X_train, gamma=0.01),
    rbf_kernel(X_test, X_train, gamma=0.1),
    poly_kernel(X_test, X_train, d=2),
    poly_kernel(X_test, X_train, d=3),
]

def combined_kernel(weights, K_list):
    return sum(weights[k] * K_list[k] for k in range(p))

# Uniform: equal weights (paper's main baseline)
mu_unif = np.ones(p) / p

results = {}
for name, mu in [('ALIGN', mu_align), ('ALIGNF', mu_alignf), ('Uniform', mu_unif)]:
    Ktr = combined_kernel(mu, K_list_train)
    Kte = combined_kernel(mu, K_list_test)
    clf = SVC(kernel='precomputed', C=1.0, random_state=42)
    clf.fit(Ktr, y_train)
    pred = clf.predict(Kte)
    acc = accuracy_score(y_test, pred)
    err = 1 - acc
    results[name] = {'acc': acc, 'err': err}
    print(f"{name:8s}: Accuracy = {acc:.4f}  |  Error = {err:.4f}")

ALIGN   : Accuracy = 0.7778  |  Error = 0.2222
ALIGNF  : Accuracy = 0.7778  |  Error = 0.2222
Uniform : Accuracy = 0.7667  |  Error = 0.2333


In [9]:
# ── Visualization ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Kernel weights comparison
x_pos = np.arange(p)
w = 0.35
axes[0].bar(x_pos - w/2, mu_align,  w, label='ALIGN',  color='steelblue', alpha=0.85)
axes[0].bar(x_pos + w/2, mu_alignf, w, label='ALIGNF', color='darkorange', alpha=0.85)
axes[0].set_xticks(x_pos); axes[0].set_xticklabels(kernel_names, rotation=15, fontsize=8)
axes[0].set_ylabel('Weight'); axes[0].set_title('Learned Kernel Weights'); axes[0].legend()

# Plot 2: Accuracy comparison
methods = list(results.keys())
accs    = [results[m]['acc'] for m in methods]
colors  = ['steelblue', 'darkorange', 'gray']
bars = axes[1].bar(methods, accs, color=colors, alpha=0.85, edgecolor='black')
axes[1].set_ylim(0.5, 1.0); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Classification Accuracy')
for bar, acc in zip(bars, accs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{acc:.3f}', ha='center', fontsize=10, fontweight='bold')

# Plot 3: Individual alignment scores
axes[2].bar(kernel_names, ind_alignments, color='teal', alpha=0.85, edgecolor='black')
axes[2].set_ylabel('Centered Alignment'); axes[2].set_title('Per-Kernel Alignment with Target')
axes[2].tick_params(axis='x', rotation=15)
for i, v in enumerate(ind_alignments):
    axes[2].text(i, v + 0.001, f'{v:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('results/task2_results.png', dpi=150, bbox_inches='tight')
plt.close()
print("Plot saved to results/task2_results.png")

Plot saved to results/task2_results.png
